# Chapter 11 — The Smallest Useful Model Call

**Companion to Applied AI**

Question: What actually changes when a model call becomes a recorded operation?

By the end of this notebook you will have:

- separated task_id, call_id, and attempt_id with assertions
- shown one logical call spanning several network attempts
- preserved the raw observation before parsing, with unknown revision left unknown

## What this notebook demonstrates
The flagship mechanism: the smallest teaching version of a recorded model call. A flaky mock provider stands in for the network; everything runs locally.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
from dataclasses import dataclass, field
import uuid, time

seed: 42


## 1. The three identities

In [2]:
@dataclass
class CallManifest:
    task_id: str
    call_id: str
    purpose: str
    created_at: float

@dataclass
class AttemptRecord:
    call_id: str
    attempt_id: str
    attempt_no: int
    outcome: str  # "ok" | "timeout" | "error"

@dataclass
class RawObservation:
    call_id: str
    attempt_id: str
    raw_bytes: bytes
    provider_revision: object = "UNKNOWN"  # never invent one

m = CallManifest(task_id="task-7", call_id="call-" + uuid.uuid4().hex[:8], purpose="classify", created_at=time.time())
a1 = AttemptRecord(m.call_id, "att-" + uuid.uuid4().hex[:8], 1, "timeout")
a2 = AttemptRecord(m.call_id, "att-" + uuid.uuid4().hex[:8], 2, "ok")
assert m.task_id != m.call_id
assert a1.attempt_id != a2.attempt_id != m.call_id
print("task:", m.task_id, "\ncall:", m.call_id, "\nattempts:", a1.attempt_no, a2.attempt_no)

task: task-7 
call: call-7310cf83 
attempts: 1 2


## 2. One logical call, two network attempts (flaky mock provider)

In [3]:
def flaky_provider(payload: dict, attempt_no: int) -> bytes:
    if attempt_no == 1:
        raise TimeoutError("response lost after the effect")
    return b'{"label": "needs_fix", "confidence": 0.81}'

attempts, raw = [], None
for no in (1, 2):
    aid = "att-" + uuid.uuid4().hex[:8]
    try:
        raw = RawObservation(m.call_id, aid, flaky_provider({"q": 1}, no))
        attempts.append(AttemptRecord(m.call_id, aid, no, "ok"))
    except TimeoutError:
        attempts.append(AttemptRecord(m.call_id, aid, no, "timeout"))
print([(a.attempt_no, a.outcome) for a in attempts])
print("raw bytes kept:", raw.raw_bytes)
print("provider revision:", raw.provider_revision)
assert len(attempts) == 2 and attempts[0].outcome == "timeout"

[(1, 'timeout'), (2, 'ok')]
raw bytes kept: b'{"label": "needs_fix", "confidence": 0.81}'
provider revision: UNKNOWN


## 3. Naive cost accounting undercounts retries

In [4]:
price_per_attempt = 0.002
naive_cost = 1 * price_per_attempt
true_cost = len(attempts) * price_per_attempt
print(f"naive (1 call): ${naive_cost:.4f}   true ({len(attempts)} attempts): ${true_cost:.4f}")
assert true_cost == 2 * naive_cost

naive (1 call): $0.0020   true (2 attempts): $0.0040


## Interpretation
- Supports: `task ≠ call ≠ attempt`; one logical call routinely spans several billable attempts; raw bytes must be stored before parsing; unknown metadata must stay `UNKNOWN`, not 0.
- Does NOT support: claims about any real provider's retry behavior.

## Try it yourself
1. Fail attempt 2 as well and add a third; watch cost triple.
2. Parse `raw.raw_bytes` with `json.loads` and show the parse reads the preserved copy.
3. Set `provider_revision = None` somewhere and watch a downstream `is UNKNOWN` check fail — that is the bug the chapter warns about.